In [1]:
from pathlib import Path
import os
import pandas as pd

# =====================================================================
#  Enertech 입력 데이터 업로드
#
#  Colab에서는 아래 4개 파일을 /content에 업로드한 뒤 실행한다.
#   1) 260902 Enertech셀 데이터 모음.xlsx
#   2) 260703 BoL 데이터 6-5.xlsx
#   3) 260703 BoL 데이터 6-6.xlsx
#   4) 260703 BoL 데이터 6-8.xlsx
# =====================================================================
DATA_DIR = Path("/content") if Path("/content").exists() else Path.cwd()

INTEGRATED_PATH = Path(os.environ.get(
    "ENERTECH_INTEGRATED_PATH",
    DATA_DIR / "260902 Enertech셀 데이터 모음.xlsx",
))
JULY_FILES = {
    "6-5": Path(os.environ.get("ENERTECH_JULY_65", DATA_DIR / "260703 BoL 데이터 6-5.xlsx")),
    "6-6": Path(os.environ.get("ENERTECH_JULY_66", DATA_DIR / "260703 BoL 데이터 6-6.xlsx")),
    "6-8": Path(os.environ.get("ENERTECH_JULY_68", DATA_DIR / "260703 BoL 데이터 6-8.xlsx")),
}

required_files = [INTEGRATED_PATH, *JULY_FILES.values()]
missing_files = [path.name for path in required_files if not path.exists()]

if missing_files:
    try:
        from google.colab import files
        print("아래 파일을 업로드하세요:")
        for name in missing_files:
            print(" -", name)
        files.upload()
    except ImportError:
        pass

missing_files = [path.name for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError("필요한 파일이 없습니다: " + ", ".join(missing_files))

required_integrated_sheets = {
    "ANODE_LOWRATE", "CATHODE_GITT", "FULLCELL_LOWRATE", "DYNAMIC"
}
integrated_sheets = set(pd.ExcelFile(INTEGRATED_PATH).sheet_names)
missing_sheets = sorted(required_integrated_sheets - integrated_sheets)
if missing_sheets:
    raise ValueError("통합 Excel에 필요한 시트가 없습니다: " + ", ".join(missing_sheets))

required_steps = {7, 10, 12, 15, 17, 20}
for cell, path in JULY_FILES.items():
    workbook = pd.ExcelFile(path)
    if not {"step", "record"}.issubset(workbook.sheet_names):
        raise ValueError(f"{path.name}: step/record 시트가 필요합니다.")
    steps = set(pd.read_excel(path, sheet_name="step", usecols=["Step Number"])["Step Number"])
    if not required_steps.issubset(steps):
        raise ValueError(f"{path.name}: 필요한 Step Number가 없습니다: {sorted(required_steps - steps)}")

print("[입력 검증] OK")
print("통합 OCP/GITT :", INTEGRATED_PATH.name)
for cell, path in JULY_FILES.items():
    print(f"July BoL {cell:>3s} : {path.name}")


[입력 검증] OK
통합 OCP/GITT : 260902 Enertech셀 데이터 모음.xlsx
July BoL 6-5 : 260703 BoL 데이터 6-5.xlsx
July BoL 6-6 : 260703 BoL 데이터 6-6.xlsx
July BoL 6-8 : 260703 BoL 데이터 6-8.xlsx


In [2]:
# =====================================================================
#  실행 환경 및 분석 코드 준비
#   - PyBaMM 26.8.0.0
#   - 공개 저장소의 검증된 Ai2020/OCP/geometry 구성 함수 사용
# =====================================================================
import shutil
import subprocess
import sys

PYBAMM_VERSION = "26.8.0.0"
REPO_BRANCH = os.environ.get("AI2020_DFN_REF", "codex/home-handoff-261008")
default_repo = Path("/content/ai2020_dfn_fitting") if Path("/content").exists() else Path.cwd()
REPO_DIR = Path(os.environ.get("AI2020_DFN_REPO", default_repo))

try:
    import pybamm
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"pybamm=={PYBAMM_VERSION}"])
    import pybamm

if pybamm.__version__ != PYBAMM_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"pybamm=={PYBAMM_VERSION}"])
    print("PyBaMM 버전이 변경됐습니다. Colab이면 런타임 재시작 후 다시 실행하세요.")

if not (REPO_DIR / ".git").exists():
    subprocess.check_call([
        "git", "clone", "--quiet", "--branch", REPO_BRANCH,
        "https://github.com/Rinny-Hi/ai2020_dfn_fitting.git",
        str(REPO_DIR),
    ])
repo_commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()

# 사용자가 업로드한 통합 Excel을 분석 코드가 읽는 표준 위치에 둔다.
repo_integrated = REPO_DIR / INTEGRATED_PATH.name
if INTEGRATED_PATH.resolve() != repo_integrated.resolve():
    shutil.copy2(INTEGRATED_PATH, repo_integrated)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("PyBaMM :", pybamm.__version__)
print("repo    : local repository")
print("commit  :", repo_commit)


PyBaMM : 26.8.0.0
repo    : local repository
commit  : aa003e9bd2d058de3672ad514043f79bf1ade8c7


In [3]:
# =====================================================================
#  [1단계] Enertech Ai2020 P0 무피팅 모델 구성
#
#  OCP/window : nominal graphite + 기존 측정 cathode
#  geometry   : common overlap area + Ai2020 Ln/Lp + SEM ImageJ Rn/Rp
#  D_s        : GITT 계산 완료값 직접 입력
#  k          : anode Ai2020 anchor + cathode EIS 계산값 직접 입력
#  hysteresis : OFF
# =====================================================================
import numpy as np
import matplotlib.pyplot as plt

import c50_selected_ocp_dynamic_validation as selected
import comprehensive_prefit_cross_cohort as cross
import electrode_hysteresis_quantification as ehq
import gitt_ocp_analysis as ga
import kinetics_original_reversion_ablation as kinetics
import prefit_physical_anchor_decision as p0
import paper_teardown_geometry as geometry
import priority_initial_state_protocol_recheck as priority

# 계산 완료된 값을 직접 사용한다. D_old/D_new 재스케일링은 하지 않는다.
LN_UM = 76.5
LP_UM = 68.0
RN_UM = 3.7542
RP_UM = 4.2387
D_S_N = 4.1707741459682473e-14
D_S_P = 6.888175323807748e-14
K_N = 9.648533212e-7
K_P = 5.899519906708763e-7
BRUGG_N = 2.914
BRUGG_P = 1.83
BRUGG_S = 1.5
EPS_S_N = 0.61
EPS_S_P = 0.62
CS_MAX_N = 25182.812491955036
CS_MAX_P = 49140.111895837676

raw_ai2020 = pybamm.ParameterValues("Ai2020")
model0, stage, anode, cathode, _, selected_ocp, qcell = selected.selected_inputs()

scenario = p0.Scenario(
    name="P0 anchored reference",
    ln_um=LN_UM,
    lp_um=LP_UM,
    rn_um=RN_UM,
    rp_um=RP_UM,
    kn=K_N,
    kp=K_P,
    dsn="measured",
    dsp="measured",
    positive_hysteresis=False,
    initialization="rest",
)
model = p0.build_model(model0, scenario)
model = geometry.apply_overlap_area(model)

# 값이 코드 경로에 의존해 바뀌지 않도록 최종 입력을 명시적으로 고정한다.
model["params"].update({
    "Negative particle diffusivity [m2.s-1]": D_S_N,
    "Positive particle diffusivity [m2.s-1]": D_S_P,
    "Negative electrode Bruggeman coefficient (electrolyte)": BRUGG_N,
    "Positive electrode Bruggeman coefficient (electrolyte)": BRUGG_P,
    "Separator Bruggeman coefficient (electrolyte)": BRUGG_S,
    "Negative electrode active material volume fraction": EPS_S_N,
    "Positive electrode active material volume fraction": EPS_S_P,
    "Maximum concentration in negative electrode [mol.m-3]": CS_MAX_N,
    "Maximum concentration in positive electrode [mol.m-3]": CS_MAX_P,
}, check_already_exists=False)
model["csn_max"] = CS_MAX_N
model["csp_max"] = CS_MAX_P
model["qn_cell"] = float(selected_ocp.Qn_Ah)
model["qp_cell"] = float(selected_ocp.Qp_Ah)

F = 96485.33212
A_OVERLAP_M2 = 45.4e-3 * 51.0e-3 * 34.0
qn_check = F * A_OVERLAP_M2 * LN_UM * 1e-6 * EPS_S_N * CS_MAX_N / 3600.0
qp_check = F * A_OVERLAP_M2 * LP_UM * 1e-6 * EPS_S_P * CS_MAX_P / 3600.0
assert np.isclose(qn_check, float(selected_ocp.Qn_Ah), atol=1e-9)
assert np.isclose(qp_check, float(selected_ocp.Qp_Ah), atol=1e-9)

parameter_table = pd.DataFrame([
    ["Ln [um]", LN_UM, "Ai2020"],
    ["Lp [um]", LP_UM, "Ai2020"],
    ["Rn [um]", RN_UM, "SEM ImageJ"],
    ["Rp [um]", RP_UM, "SEM ImageJ"],
    ["overlap area [cm2]", A_OVERLAP_M2 * 1e4, "45.4 mm x 51.0 mm x 34 faces"],
    ["Dsn [m2/s]", D_S_N, "all-range apparent GITT"],
    ["Dsp [m2/s]", D_S_P, "all-range apparent GITT"],
    ["kn prefactor", K_N, "Ai2020 anchor"],
    ["kp prefactor", K_P, "Cathode P4 EIS apparent"],
    ["eps_s,n", EPS_S_N, "teardown-literature estimate"],
    ["eps_s,p", EPS_S_P, "teardown-literature estimate"],
    ["csn,max [mol/m3]", CS_MAX_N, "capacity-consistent effective inverse"],
    ["csp,max [mol/m3]", CS_MAX_P, "capacity-consistent effective inverse"],
    ["brugg_n", BRUGG_N, "Ai2020"],
    ["brugg_p", BRUGG_P, "Ai2020"],
    ["brugg_s", BRUGG_S, "Ai2020"],
], columns=["Parameter", "Value", "Source"])

print("=" * 78)
print("[Enertech Ai2020 P0 parameter set — fitting 없음]")
print(parameter_table.to_string(index=False))
print("\nOCP window")
print(stage)
print("Qcell = %.9f Ah" % qcell)
print("Qn/Qp = %.7f / %.7f Ah" % (float(selected_ocp.Qn_Ah), float(selected_ocp.Qp_Ah)))
print("csn,max/csp,max = %.1f / %.1f mol/m3" % (
    float(model["params"]["Maximum concentration in negative electrode [mol.m-3]"]),
    float(model["params"]["Maximum concentration in positive electrode [mol.m-3]"]),
))
print("=" * 78)


[Enertech Ai2020 P0 parameter set — fitting 없음]
         Parameter        Value                                Source
           Ln [um] 7.650000e+01                                Ai2020
           Lp [um] 6.800000e+01                                Ai2020
           Rn [um] 3.754200e+00                            SEM ImageJ
           Rp [um] 4.238700e+00                            SEM ImageJ
overlap area [cm2] 7.872360e+02          45.4 mm x 51.0 mm x 34 faces
        Dsn [m2/s] 4.170774e-14               all-range apparent GITT
        Dsp [m2/s] 6.888175e-14               all-range apparent GITT
      kn prefactor 9.648533e-07                         Ai2020 anchor
      kp prefactor 5.899520e-07               Cathode P4 EIS apparent
           eps_s,n 6.100000e-01          teardown-literature estimate
           eps_s,p 6.200000e-01          teardown-literature estimate
  csn,max [mol/m3] 2.518281e+04 capacity-consistent effective inverse
  csp,max [mol/m3] 4.914011e+04 capacity-c

In [4]:
# =====================================================================
#  [2단계] July BoL 3셀의 0.5C/1C/2C charge/discharge 데이터 구성
#   - Step Number는 원본 RPT protocol 그대로 사용
#   - 각 branch 직전 rest 종점전압으로 초기 SOC를 계산
# =====================================================================
OLD_BRANCH_STEPS = {
    (0.5, True): 7,
    (0.5, False): 10,
    (1.0, True): 12,
    (1.0, False): 15,
    (2.0, True): 17,
    (2.0, False): 20,
}

curves = {}
qc_rows = []
for cell, path in JULY_FILES.items():
    cell_curves, rows = cross.parse_cell(
        "Old July BoL", cell, path, OLD_BRANCH_STEPS
    )
    curves[cell] = cell_curves
    qc_rows.extend(rows)

qc = pd.DataFrame(qc_rows)
protocol = cross.cohort_protocol_means(qc)

qc_view = qc[[
    "Cell", "C_rate", "Direction", "Step_number", "Rest_end_V",
    "Median_current_A", "CC_duration_min", "CC_capacity_Ah"
]].sort_values(["C_rate", "Direction", "Cell"])

print("[July BoL 입력 branch 확인]")
print(qc_view.to_string(index=False))
print("\n[3셀 평균 protocol]")
print(protocol.to_string())


[July BoL 입력 branch 확인]
Cell  C_rate Direction  Step_number  Rest_end_V  Median_current_A  CC_duration_min  CC_capacity_Ah
 6-5     0.5    Charge            7      3.0542          1.139889       112.783333        2.142673
 6-6     0.5    Charge            7      3.0601          1.139948       112.933333        2.145856
 6-8     0.5    Charge            7      3.0580          1.139996       113.116667        2.149239
 6-5     0.5 Discharge           10      4.1780          1.139888       120.516667        2.289720
 6-6     0.5 Discharge           10      4.1783          1.139974       120.533333        2.290367
 6-8     0.5 Discharge           10      4.1788          1.139903       120.500000        2.289748
 6-5     1.0    Charge           12      3.2077          2.279909        49.983333        1.899352
 6-6     1.0    Charge           12      3.2073          2.279965        50.233333        1.908774
 6-8     1.0    Charge           12      3.2055          2.279986        50.533333   

In [5]:
# =====================================================================
#  [3단계] 0.5C/1C/2C 무피팅 DFN baseline
#   - charge/discharge 모두 같은 고정 parameter set 사용
#   - 각 rate의 평균 실측전류와 rest 기반 초기 SOC 사용
#   - capacity와 전압을 별도로 평가
# =====================================================================
simulations = {}
metric_rows = []

for rate in ga.RATES:
    for charge in (True, False):
        direction = "Charge" if charge else "Discharge"
        p = protocol.loc[("Old July BoL", rate, direction)]
        z0 = priority.monotone_voltage_inverse(
            model["v_qocv"], model["soc"], float(p.Rest_end_V)
        )
        local_stage = priority.stage_at_soc(stage, z0, charge)
        simulation_rate = float(p.Measured_current_A) / cross.NOMINAL_CAPACITY_AH

        print(f"{rate:g}C {direction} | initial SOC={100*z0:.3f}% | simulation rate={simulation_rate:.5f}C")
        simulation = ehq.run_dfn(
            simulation_rate,
            charge,
            local_stage,
            model,
            anode,
            cathode,
            False,
            False,
        )
        simulations[(rate, charge)] = simulation

        for cell, branches in curves.items():
            metrics = priority.curve_metrics(
                branches[(rate, charge)], simulation, qcell, charge
            )
            metric_rows.append({
                "Scenario": "P0 anchored reference",
                "Trust": "anchored",
                "Note": "",
                "Cell": cell,
                "C_rate": rate,
                "Direction": direction,
                **metrics,
                "Capacity_error_mAh": 1000.0 * (
                    metrics["Q_end_model_Ah"] - metrics["Q_end_exp_Ah"]
                ),
            })

detail = pd.DataFrame(metric_rows)
summary = p0.summarize(detail)
branch_summary = p0.branch_summary(detail, ["P0 anchored reference"])

print("\n" + "=" * 100)
print("[Enertech Ai2020 P0 — fitting 전 최종 결과]")
print(summary.to_string(index=False))
print("\n[C-rate / direction별 결과]")
print(branch_summary.to_string(index=False))
print("=" * 100)

fig, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)
for row, charge in enumerate((True, False)):
    direction = "Charge" if charge else "Discharge"
    for col, rate in enumerate(ga.RATES):
        ax = axes[row, col]
        for idx, (cell, branches) in enumerate(curves.items()):
            exp = branches[(rate, charge)]
            ax.plot(exp["Q_Ah"], exp["V"], color="black", alpha=0.30,
                    label="Experiment, n=3" if idx == 0 else None)
        sim = simulations[(rate, charge)]
        ax.plot(sim["Q_Ah"], sim["V"], color="#0072B2", lw=2.2, label="Ai2020 P0")
        ax.set(title=f"{rate:g}C {direction}", xlabel="Capacity [Ah]", ylabel="Voltage [V]")
        ax.grid(alpha=0.2)
        ax.legend(fontsize=8)
plt.show()


0.5C Charge | initial SOC=0.089% | simulation rate=0.49998C


0.5C Discharge | initial SOC=98.425% | simulation rate=0.49997C


1C Charge | initial SOC=0.543% | simulation rate=0.99998C


1C Discharge | initial SOC=98.391% | simulation rate=0.99997C


2C Charge | initial SOC=1.155% | simulation rate=1.99993C


2C Discharge | initial SOC=98.459% | simulation rate=1.99996C



[Enertech Ai2020 P0 — fitting 전 최종 결과]
             Scenario    Trust Note  Full_MAE_mV  Full_RMSE_mV  Center10_70_MAE_mV  Center10_70_RMSE_mV  Capacity_RMSE_pct  Mean_abs_capacity_error_mAh  Max_abs_capacity_error_mAh  Pareto_center_capacity
P0 anchored reference anchored         34.681402      42.55733           31.528199            33.454515           1.001045                    16.367821                   31.923773                    True

[C-rate / direction별 결과]
             Scenario  C_rate Direction  Full_RMSE_mV  Center10_70_RMSE_mV  Center10_70_MAE_mV  Capacity_error_pct  Capacity_error_mAh
P0 anchored reference     0.5    Charge     42.076924            47.354613           45.354325           -0.925926          -19.872980
P0 anchored reference     0.5 Discharge     43.201821            24.590280           22.320884            0.526358           12.053264
P0 anchored reference     1.0    Charge     48.090189            42.297524           40.517168            0.042325       

In [6]:
# =====================================================================
#  [4단계] 피팅 전 후보 비교와 최종 선택
#   - 수치 오차만 가장 작은 legacy 조합은 선택하지 않는다.
#   - 독립 물리 근거와 전압/용량 균형이 있는 P0를 fitting 시작점으로 사용한다.
#   - 이 노트북에서는 parameter fitting을 실행하지 않는다.
# =====================================================================
stored_summary = pd.read_csv(
    REPO_DIR / "results/260927_prefit_physical_anchor_decision/prefit_anchor_summary.csv"
)

current_p0 = summary.iloc[0]
stored_p0 = stored_summary[stored_summary["Scenario"] == "P0 anchored reference"].iloc[0]
print("[현재 기준 재계산] OK - 2~98% window, common-overlap area, updated GITT/EIS inputs")
print("\n[과거 후보표: 비교용]")
anchored = stored_summary[stored_summary["Trust"] == "anchored"].copy()
print(anchored[[
    "Scenario", "Full_RMSE_mV", "Center10_70_RMSE_mV",
    "Capacity_RMSE_pct", "Mean_abs_capacity_error_mAh", "Pareto_center_capacity"
]].to_string(index=False))

fig, ax = plt.subplots(figsize=(9.5, 6.2))
colors = stored_summary["Trust"].map({
    "anchored": "#1f77b4",
    "literature comparator": "#2ca02c",
    "sensitivity-only": "#ff7f0e",
    "invalidated numerical comparator": "#d62728",
    "method comparator": "#9467bd",
}).fillna("#777777")
ax.scatter(stored_summary["Capacity_RMSE_pct"], stored_summary["Center10_70_RMSE_mV"],
           c=colors, s=65)
for _, row in stored_summary.iterrows():
    ax.annotate(row["Scenario"].split()[0],
                (row["Capacity_RMSE_pct"], row["Center10_70_RMSE_mV"]),
                xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.scatter([current_p0["Capacity_RMSE_pct"]], [current_p0["Center10_70_RMSE_mV"]],
           s=190, facecolors="none", edgecolors="red", linewidths=2, label="Current 2-98% P0")
ax.set(xlabel="Capacity RMSE [%]", ylabel="10-70% voltage RMSE [mV]",
       title="July BoL pre-fit candidates")
ax.grid(alpha=0.25)
ax.legend()
plt.show()

print("\n최종 무피팅 기준: current 2-98% P0")
print("  Full MAE / RMSE       = %.2f / %.2f mV" % (current_p0.Full_MAE_mV, current_p0.Full_RMSE_mV))
print("  10-70%% MAE / RMSE    = %.2f / %.2f mV" % (current_p0.Center10_70_MAE_mV, current_p0.Center10_70_RMSE_mV))
print("  Capacity RMSE         = %.2f %%" % current_p0.Capacity_RMSE_pct)
print("  Mean / max |cap error|= %.2f / %.2f mAh" % (
    current_p0.Mean_abs_capacity_error_mAh, current_p0.Max_abs_capacity_error_mAh
))
print("\n여기까지가 parameter fitting 전 결과입니다.")


[현재 기준 재계산] OK - 2~98% window, common-overlap area, updated GITT/EIS inputs

[과거 후보표: 비교용]
                         Scenario  Full_RMSE_mV  Center10_70_RMSE_mV  Capacity_RMSE_pct  Mean_abs_capacity_error_mAh  Pareto_center_capacity
P1 anchored + positive hysteresis     37.892391            25.150529           1.259103                    17.819672                    True
                   D0 nominal Dsn     37.507227            22.665373           1.267833                    20.092247                    True
            P0 anchored reference     37.947179            21.916628           1.315739                    18.708800                    True
                  R0 Ai2020 radii     39.653442            24.002075           1.480271                    23.924046                   False
                 L1 SEM thickness     42.495436            28.372297           1.538973                    15.535281                   False
     P2 anchored + endpoint start     40.321039            25.8